# Apply LSTM model to hydro data

Use HydroDF csv was was compiled in [LSTM_data](LSTM_data.ipynb). Use torch310 environment.

In [3]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import joblib

from supportingscripts import LSTM_helper

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In this application, the LSTM uses a 30-day lookback window of meteorological and hydrologic inputs to predict daily streamflow. By learning relationships across time, the model can represent processes such as snowmelt timing, precipitation-runoff response, and seasonal dynamics, making it well-suited for hydrologic forecasting tasks.


Split rule used here:
- Training: 2006–2014*********UPDATE
- Validation: 2015–2018******UPDATE
- Testing: 2019–2021********UPDATE

In [ ]:
# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Data parameters
FILE_PATH = "data/HydroDF/HydroDF_TuolumneRiverBasin_11274790.csv"
DATE_COL = "Date"
TARGET_COL = "flow_cms"

# Hyperparameters
LOOKBACK_DAYS = 30 #number of past time steps to use for prediction
BATCH_SIZE = 64 #number of samples per batch for training
EPOCHS = 50 #number of times to iterate over the entire training dataset
PATIENCE = 8 #number of epochs to wait for improvement in validation loss before stopping training
LEARNING_RATE = 1e-3 #step size for updating model parameters during training

# Define the years for splitting the dataset into training, validation, and testing sets
TRAIN_START_YEAR = 1980
TRAIN_END_YEAR = 2014
VAL_START_YEAR = 2015
VAL_END_YEAR = 2018
TEST_START_YEAR = 2019
TEST_END_YEAR = 2021